In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark import SparkFiles
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.regression import LinearRegression

os.environ["HADOOP_HOME"] = r"C:\hadoop\winutils\hadoop-3.3.6"
os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]

# Inicializar sesión
spark = SparkSession.builder.appName("RegresionStartups").getOrCreate()

# Cargar el dataset de 50 Startups
url_startups = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv"
spark.sparkContext.addFile(url_startups)

df_startups = spark.read.csv(SparkFiles.get("50_Startups.csv"), header=True, inferSchema=True)

# Limpiamos nombres de columnas para evitar problemas con los espacios
df_startups = df_startups.withColumnRenamed("R&D Spend", "RD_Spend") \
.withColumnRenamed("Marketing Spend", "Marketing_Spend")

df_startups.show(5)

+---------+--------------+---------------+----------+---------+
| RD_Spend|Administration|Marketing_Spend|     State|   Profit|
+---------+--------------+---------------+----------+---------+
| 165349.2|      136897.8|       471784.1|  New York|192261.83|
| 162597.7|     151377.59|      443898.53|California|191792.06|
|153441.51|     101145.55|      407934.54|   Florida|191050.39|
|144372.41|     118671.85|      383199.62|  New York|182901.99|
|142107.34|      91391.77|      366168.42|   Florida|166187.94|
+---------+--------------+---------------+----------+---------+
only showing top 5 rows


In [3]:
# Preparar la variable objetivo y las características en formato Vector
assembler = VectorAssembler(
    inputCols=["RD_Spend"],
    outputCol="features"
)

df_simple = assembler.transform(df_startups)
df_simple.show(5)

+---------+--------------+---------------+----------+---------+-----------+
| RD_Spend|Administration|Marketing_Spend|     State|   Profit|   features|
+---------+--------------+---------------+----------+---------+-----------+
| 165349.2|      136897.8|       471784.1|  New York|192261.83| [165349.2]|
| 162597.7|     151377.59|      443898.53|California|191792.06| [162597.7]|
|153441.51|     101145.55|      407934.54|   Florida|191050.39|[153441.51]|
|144372.41|     118671.85|      383199.62|  New York|182901.99|[144372.41]|
|142107.34|      91391.77|      366168.42|   Florida|166187.94|[142107.34]|
+---------+--------------+---------------+----------+---------+-----------+
only showing top 5 rows


In [4]:
# Dividir los datos en conjunto de entrenamiento y pruebas
train_data, test_data = df_simple.randomSplit([0.8, 0.2], seed=42)
print("Train Data Count: ", train_data.count())
print("Test Data Count: ", test_data.count())

# Inicia el entrenamiento de regresion lineal
lr_simple = LinearRegression(featuresCol="features", labelCol="Profit")
lr_model = lr_simple.fit(train_data)

# Evaluar el modelo en el conjunto de prueba
test_results = lr_model.evaluate(test_data)
print("RMSE: ", test_results.rootMeanSquaredError)
print("R2: ", test_results.r2)

# Mostrar las predicciones
test_results.predictions.show(5)

Train Data Count:  38
Test Data Count:  12
RMSE:  9229.715324219018
R2:  0.9645939866432084
+--------+--------------+---------------+----------+--------+----------+------------------+
|RD_Spend|Administration|Marketing_Spend|     State|  Profit|  features|        prediction|
+--------+--------------+---------------+----------+--------+----------+------------------+
|  542.05|      51743.15|            0.0|  New York|35673.41|  [542.05]| 50427.33354978463|
|20229.59|      65947.93|       185265.1|  New York|81229.06|[20229.59]| 66772.25497517604|
|23640.93|      96189.63|      148001.11|California|71498.49|[23640.93]| 69604.40588155107|
|44069.95|      51283.14|      197029.42|California|89949.14|[44069.95]| 86564.91627024104|
|63408.86|     129219.61|       46085.25|California|97427.84|[63408.86]|102620.39930177855|
+--------+--------------+---------------+----------+--------+----------+------------------+
only showing top 5 rows


In [6]:
# Ejercicio propuesto: Construir modelo de regresion extra, que considere tres variables de gasto (Spend) como predictoras (RD_Spend, Marketing_Spend, Administration)

assembler_extra = VectorAssembler(
    inputCols=["RD_Spend", "Marketing_Spend", "Administration"],
    outputCol="features"
)

df_extra = assembler_extra.transform(df_startups)
df_extra.show(5)

# Dividir los datos en conjunto de entrenamiento y pruebas
train_data_extra, test_data_extra = df_extra.randomSplit([0.8, 0.2], seed=42)
print("Train Data Count: ", train_data_extra.count())
print("Test Data Count: ", test_data_extra.count())

# Inicia el entrenamiento de regresion lineal
lr_extra = LinearRegression(featuresCol="features", labelCol="Profit")
lr_model_extra = lr_extra.fit(train_data_extra)

# Evaluar el modelo en el conjunto de prueba
test_results_extra = lr_model_extra.evaluate(test_data_extra)
print("RMSE: ", test_results_extra.rootMeanSquaredError)
print("R2: ", test_results_extra.r2)

# Coeficientes asignados a las variables predictoras
print("Coeficientes: ", lr_model_extra.coefficients)
print("Intercepto: ", lr_model_extra.intercept)

# Mostrar las predicciones
test_results_extra.predictions.show(5)

+---------+--------------+---------------+----------+---------+--------------------+
| RD_Spend|Administration|Marketing_Spend|     State|   Profit|            features|
+---------+--------------+---------------+----------+---------+--------------------+
| 165349.2|      136897.8|       471784.1|  New York|192261.83|[165349.2,471784....|
| 162597.7|     151377.59|      443898.53|California|191792.06|[162597.7,443898....|
|153441.51|     101145.55|      407934.54|   Florida|191050.39|[153441.51,407934...|
|144372.41|     118671.85|      383199.62|  New York|182901.99|[144372.41,383199...|
|142107.34|      91391.77|      366168.42|   Florida|166187.94|[142107.34,366168...|
+---------+--------------+---------------+----------+---------+--------------------+
only showing top 5 rows
Train Data Count:  38
Test Data Count:  12
RMSE:  8286.260695094332
R2:  0.971462389786215
Coeficientes:  [0.809077417206832,0.014714607903703355,-0.038212063764521784]
Intercepto:  53450.50889180107
+--------+-

## Guía didáctica: cómo elegir un modelo de Machine Learning

### 1. Empieza por la variable objetivo

Antes de elegir un algoritmo, identifica qué tipo de dato quieres predecir:

| Variable objetivo | Tipo de problema | Ejemplo de modelo |
|---|---|---|
| Número continuo | Regresión | `LinearRegression`, `RandomForestRegressor` |
| Clase o categoría | Clasificación | `LogisticRegression`, `RandomForestClassifier` |
| Valor futuro ordenado por fecha | Pronóstico temporal | modelos temporales o regresores con variables históricas |

En este ejercicio, `Profit` es un número continuo. Por eso usamos un modelo de **regresión**. No estamos prediciendo si una startup pertenece a una categoría, sino cuánto beneficio puede obtener.

### 2. Diferencia entre `features`, `label` y `prediction`

En el dataset de startups:

```text
features = RD_Spend, Marketing_Spend, Administration
label    = Profit real
prediction = Profit estimado por el modelo
```

`VectorAssembler` reúne las variables predictoras en una columna vectorial:

```python
assembler_extra = VectorAssembler(
    inputCols=["RD_Spend", "Marketing_Spend", "Administration"],
    outputCol="features"
)
```

`Profit` no debe incluirse en `inputCols`, porque es precisamente la respuesta que queremos predecir. Incluirlo produciría **fuga de información**: el modelo recibiría como entrada el valor que debe aprender a estimar.

### 3. Qué aprende la regresión lineal

La regresión lineal busca los coeficientes que mejor aproximan los datos mediante una ecuación:

$$
Profit_{estimado} = \beta_0 + \beta_1 RD\_Spend + \beta_2 Marketing\_Spend + \beta_3 Administration
$$

- $\beta_0$ es el intercepto.
- Cada $\beta$ mide la relación estimada entre una variable y `Profit`.
- El modelo busca que las predicciones estén lo más cerca posible de los valores reales.

Con una variable predictora se habla de una recta. Con tres variables predictoras, la representación geométrica es un hiperplano.

Una ventaja importante es la interpretación. Después del entrenamiento se pueden consultar los coeficientes:

```python
print("Intercepto:", lr_model_extra.intercept)
print("Coeficientes:", lr_model_extra.coefficients)
```

Estos coeficientes deben interpretarse con cuidado: expresan asociaciones aprendidas a partir de los datos y no demuestran por sí solos causalidad.

### 4. Por qué la regresión lineal es un buen primer modelo

La regresión lineal se utiliza aquí como **baseline**, es decir, como modelo de referencia:

- `Profit` es continuo y el algoritmo está diseñado para regresión.
- El dataset es pequeño, con aproximadamente 50 observaciones.
- Es rápida de entrenar.
- Es sencilla de explicar a una persona de negocio.
- Permite inspeccionar directamente la influencia estimada de cada variable.
- Sirve para comprobar si una relación lineal ya explica razonablemente los datos.

Un baseline no tiene que ser el modelo definitivo. Su valor es proporcionar una referencia contra la que comparar modelos más complejos.

### 5. Cuándo probar `RandomForestRegressor`

Un Random Forest también puede predecir `Profit`, pero debe utilizarse la variante de regresión:

```python
from pyspark.ml.regression import RandomForestRegressor

rf_reg = RandomForestRegressor(
    featuresCol="features",
    labelCol="Profit",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_model = rf_reg.fit(train_data_extra)
rf_predictions = rf_model.transform(test_data_extra)
rf_predictions.select("Profit", "prediction").show(10)
```

`RandomForestRegressor` puede ser útil si la relación entre gastos y beneficio no es lineal o si existen interacciones entre variables. Por ejemplo, el efecto del marketing podría depender del nivel de inversión en investigación y desarrollo.

El bosque puede aprender reglas como:

```text
si RD_Spend es alto y Marketing_Spend es medio -> predicción A
si RD_Spend es bajo y Administration es alto  -> predicción B
```

No necesita que la relación siga una única recta.

### 6. Comparación entre ambos modelos

La comparación debe realizarse con los mismos datos de prueba y las mismas métricas:

| Característica | Regresión lineal | Random Forest regresor |
|---|---|---|
| Forma aprendida | Relación lineal | Reglas y relaciones no lineales |
| Interpretación | Alta | Menor |
| Coste computacional | Bajo | Mayor |
| Riesgo con pocos datos | Generalmente bajo | Puede sobreajustar |
| Extrapolación | Puede extrapolar una tendencia | No extrapola bien fuera del rango observado |
| Hiperparámetros | Pocos | Muchos |

Métricas habituales:

- `RMSE`: error medio expresado en las unidades de `Profit`; cuanto menor, mejor.
- `R2`: proporción de variabilidad explicada; cuanto mayor, mejor, aunque puede ser engañosa con pocos datos.
- `MAE`: error absoluto medio, menos sensible a valores extremos que `RMSE`.

No se debe elegir automáticamente el modelo más complejo. Hay que comprobar si la mejora en las métricas justifica perder interpretabilidad y aumentar el coste.

### 7. Entrenamiento, validación y prueba

Una práctica habitual separa los datos en tres conjuntos:

```text
train      -> aprende parámetros y reglas
validation -> ayuda a elegir hiperparámetros
test       -> evaluación final sobre datos no vistos
```

En este ejemplo se utiliza `train` y `test` por simplicidad. En un proyecto real, especialmente con pocos datos, puede ser más apropiada la validación cruzada.

Es importante no elegir el modelo mirando repetidamente el resultado del conjunto de prueba. Si se toma una decisión después de observar muchas veces el test, este deja de ser una evaluación realmente independiente.

### 8. Cómo detectar sobreajuste

Un modelo está sobreajustado cuando aprende demasiado bien los datos de entrenamiento, incluidos ruido y casos particulares, pero funciona mal con datos nuevos.

Una señal típica es:

```text
RMSE de entrenamiento muy bajo
RMSE de prueba mucho más alto
```

En Random Forest, una profundidad excesiva puede crear reglas demasiado específicas. En regresión lineal, el riesgo puede aparecer por variables irrelevantes, fuga de información o una evaluación poco representativa.

La solución no es siempre aumentar la complejidad. Puede ser necesario:

- Reducir `maxDepth`.
- Aumentar `minInstancesPerNode`.
- Eliminar variables irrelevantes.
- Aumentar la cantidad de datos.
- Utilizar validación cruzada.
- Revisar que no exista fuga de información.

### 9. Flujo mental de un developer de ML

Antes de escribir el modelo, responde estas preguntas:

1. ¿Qué variable quiero predecir?
2. ¿Es continua, categórica o temporal?
3. ¿Qué información estaría disponible en el momento de la predicción?
4. ¿Qué columnas son predictoras y cuál es la etiqueta?
5. ¿Qué modelo simple sirve como baseline?
6. ¿Qué métrica representa mejor el objetivo del negocio?
7. ¿Cómo separaré entrenamiento, validación y prueba?
8. ¿Cómo detectaré sobreajuste y fuga de información?
9. ¿La mejora de un modelo complejo compensa su menor interpretabilidad?

La idea principal es que elegir un modelo no consiste en buscar el algoritmo más sofisticado, sino en relacionar correctamente el tipo de problema, los datos disponibles, la métrica, el coste operativo y la necesidad de interpretar las predicciones.